# 04 — Yahoo Auctions JP (phase 0)

Prove vehicle search against Yahoo's official API (`auccat=26360`), map response fields, and run the proxy recon checklist.

Setup guide: [`docs/yahoo-setup.md`](../docs/yahoo-setup.md)  
Full plan: [`docs/plans/yahoo-proxy-ingestion.md`](../docs/plans/yahoo-proxy-ingestion.md)

Run: `make jupyter` → open this notebook, or local `jupyter lab notebooks/`.

In [ ]:
from notification_rake.config import settings
from notification_rake.ingestion.yahoo import (
    AUCCAT_USED_NEW_CARS,
    YahooClient,
    search_vehicle_auctions,
)

print("YAHOO_APP_ID set:", bool(settings.yahoo_app_id))
print("Vehicle auccat:", settings.yahoo_vehicle_auccat, "(expected", AUCCAT_USED_NEW_CARS, ")")
print("API base:", settings.yahoo_auction_api_base)
print("Daily budget:", settings.yahoo_requests_per_day)

In [ ]:
# Live API when YAHOO_APP_ID is set; otherwise bundled fixture
client = YahooClient.from_settings()
result = search_vehicle_auctions(query="スカイライン", page=1, client=client)

print(f"total_available={result.total_available} returned={len(result.items)}")
print(f"client requests today: {client.stats.requests_today}")
result.sample_fields()

In [ ]:
from notification_rake.ingestion.yahoo import iter_vehicle_search_pages

pages = iter_vehicle_search_pages(query="トヨタ", max_pages=3, client=client)
for p in pages:
    print(f"page {p.page}: {len(p.items)} items (total ~{p.total_available})")

print("cumulative requests:", client.stats.requests_today)

In [ ]:
import json
from pathlib import Path

# Mapped fields for ingest schema design
if result.items:
    print(json.dumps(result.items[0].field_map, indent=2, ensure_ascii=False))

# First raw item keys (live API or fixture)
raw_items = (
    (result.raw.get("ResultSet") or {}).get("Result") or {}
).get("Item")
if isinstance(raw_items, dict):
    raw_items = [raw_items]
if raw_items:
    print("raw keys:", sorted(raw_items[0].keys()))

print("Proxy endpoint template:", Path("../docs/plans/yahoo-proxy-endpoints.json").resolve())

## Proxy recon checklist (manual)

For each site, open DevTools → Network → filter Fetch/XHR, then search **スカイライン** in Yahoo auctions (vehicles / `26360`):

| Proxy | URL |
|-------|-----|
| Buyee | https://buyee.jp/ |
| Neokyo | https://neokyo.com/ |
| FromJapan | https://www.fromjapan.co.jp/ |

Log URL, method, params, and whether `AuctionID` matches Yahoo. Paste into `docs/plans/yahoo-proxy-endpoints.json`.

Phase 1 next: wire `search_vehicle_auctions` → `store_raw_listings` → pipeline.